Этап 1. Проведите исследовательский анализ (EDA)
Загрузите датасет и визуализируйте часть данных. Изучите то, с чем предстоит работать
Сформируйте видение:
как вы будете решать задачу,
какие подходы к обогащению/аугментации данных примените и почему,
на какие метрики будете ориентироваться при обучении.
Результаты: EDA и выводы о подходах к решению задачи. 

Подготовка DataFrame

In [1]:
!pip install pandas albumentations timm transformers pillow
import pandas as pd

# Загружаем CSV
dish_df = pd.read_csv("data/dish.csv")
ingr_df = pd.read_csv("data/ingredients.csv")

# Словарь: id -> название ингредиента
ingr_map = dict(zip(
    ingr_df["id"].astype(str).str.zfill(10),
    ingr_df["ingr"]
))

def ingredients_to_text(ingr_string):
    ids = ingr_string.split(";")
    names = []
    for ingr_id in ids:
        num_id = ingr_id.replace("ingr_", "")
        if num_id in ingr_map:
            names.append(ingr_map[num_id])
    return ", ".join(names)

# Формируем текст
dish_df["text"] = dish_df["ingredients"].apply(ingredients_to_text)

# Пути к изображениям
dish_df["image_path"] = dish_df["dish_id"].apply(
    lambda x: f"{x}/rgb.png"
)

# Label = калории (REGRESSION!)
dish_df["label"] = dish_df["total_calories"]

# Train / test
train_df = dish_df[dish_df["split"] == "train"].reset_index(drop=True)
test_df  = dish_df[dish_df["split"] == "test"].reset_index(drop=True)


  Using cached pandas-3.0.0-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached timm-1.0.24-py3-none-any.whl.metadata (38 kB)
  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)
  Using cached pillow-12.1.0-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached numpy-2.4.2-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.17.0-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.13.0.90-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached stringzilla-4.6.0-cp311-cp311-win_amd64.whl.metadata (124 kB)
  Using cached simsimd-6.5.12-cp311-cp311-win_amd64.whl.metadata (71 kB)
  Usi

Dataset

In [2]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import timm
from transformers import AutoTokenizer

class MultimodalDataset(Dataset):
    def __init__(self, df, text_model, image_model, transforms):
        self.df = df
        self.image_cfg = timm.get_pretrained_cfg(image_model)
        self.tokenizer = AutoTokenizer.from_pretrained(text_model)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        text = row["text"]
        label = torch.tensor(row["label"], dtype=torch.float32)

        img_path = f"data/images/{row['image_path']}"
        image = Image.open(img_path).convert("RGB")
        image = self.transforms(image=np.array(image))["image"]

        return {
            "text": text,
            "image": image,
            "label": label
        }


c:\Users\Admin\NEIRO\SPR_4_FINAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collate fn

In [3]:
def collate_fn(batch, tokenizer):
    texts = [item["text"] for item in batch]
    images = torch.stack([item["image"] for item in batch])
    labels = torch.stack([item["label"] for item in batch])

    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    return {
        "image": images,
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "label": labels
    }


Аугментации

In [4]:
import albumentations as A

image_model = "tf_efficientnet_b0"
cfg = timm.get_pretrained_cfg(image_model)

transforms = A.Compose([
    A.SmallestMaxSize(max_size=max(cfg.input_size[1], cfg.input_size[2])),
    A.RandomCrop(cfg.input_size[1], cfg.input_size[2]),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(0.2, 0.2, 0.2, 0.1, p=0.7),
    A.Normalize(mean=cfg.mean, std=cfg.std),
])


DataLoaders

In [5]:
from torch.utils.data import DataLoader
from functools import partial

text_model = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(text_model)

train_ds = MultimodalDataset(
    train_df,
    text_model=text_model,
    image_model=image_model,
    transforms=transforms
)

test_ds = MultimodalDataset(
    test_df,
    text_model=text_model,
    image_model=image_model,
    transforms=transforms
)

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=partial(collate_fn, tokenizer=tokenizer)
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=partial(collate_fn, tokenizer=tokenizer)
)
